In [74]:
import yaml
from types import SimpleNamespace
# from strategies.run_allen import get_allen_result, get_allen_signals
from strategies.run_gino import get_gino_result, get_gino_signals
from strategies.utils.analysis import get_equal_weight_baseline_result
from utils.io import read_all_df
import pandas as pd
from evaluation.stats import sharpe
import finlab
from finlab import data

finlab.login('ntSS3778pZi2FfkeYxXP0p+S0iI4AggkcphAUxh/lTVrWqT2FreKQsDkTA92CM7d#vip_m')

CATEGORY = 'Selected'

result_dir = f"results/Original/{CATEGORY}_Dataset_Abs_2025"
# NOTE: the result_dir should only contain results of 2025


def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)
backtest_config = load_config("config/backtest.yaml")

f = open("config/backtest.yaml")
cfg = yaml.safe_load(f)

result = get_gino_result(cfg, result_dir)
signals = get_gino_signals(cfg, result_dir)
test_pred, test_truth, _, _ = read_all_df(result_dir)

輸入成功!
Exceeded maximum positions of 30 at 2025-02-18 00:00:00 for symbol 3019!
Exceeded maximum positions of 30 at 2025-02-19 00:00:00 for symbol 3019!
Exceeded maximum positions of 30 at 2025-02-20 00:00:00 for symbol 3019!
Exceeded maximum positions of 30 at 2025-02-21 00:00:00 for symbol 3019!
Exceeded maximum positions of 30 at 2025-02-24 00:00:00 for symbol 3019!
Not Enough Money to buy at 2025-05-06 00:00:00 for stock 2408!
Not Enough Money to buy at 2025-05-07 00:00:00 for stock 1514!
Not Enough Money to buy at 2025-07-30 00:00:00 for stock 3037!
Not Enough Money to buy at 2025-08-12 00:00:00 for stock 8046!


In [75]:
baseline_results = get_equal_weight_baseline_result(result_dir, pd.to_datetime('2025-01-01'), pd.Timestamp.now().strftime("%Y-%m-%d"))

In [76]:
def yearly_returns(returns):
    yearly_roi = returns.resample("Y").last() / returns.resample("Y").first() - 1
    rois = {}
    for year in yearly_roi.index:
        rois[year.year] = yearly_roi[year]
    rois['final'] = returns.iloc[-1] / returns.iloc[0] - 1
    rois['sharpe'] = sharpe(returns)
    return rois

In [77]:
model_rois = yearly_returns(result['model'].returns)
baseline_rois = yearly_returns(baseline_results.returns)

assert model_rois.keys() == baseline_rois.keys(), 'model should only contain 2025'

print("model rois =", model_rois)
print("baseline rois =", baseline_rois)

model rois = {2025: 0.23969328326484618, 'final': 0.23969328326484618, 'sharpe': 0.993658953288424}
baseline rois = {2025: 0.12368186827929706, 'final': 0.12368186827929706, 'sharpe': 0.7412408664346664}


In [78]:
from evaluation.surge_metrics import compute_surge_metrics_from_buy_signals

model_data = {
    'category': CATEGORY,
    'baseline': 0,
}
model_data.update(model_rois)

baseline_data = {
    'category': CATEGORY,
    'baseline': 1,
}
baseline_data.update(baseline_rois)

result_df = pd.DataFrame([model_data, baseline_data])
result_df


,category,baseline,2025,final,sharpe
0,Selected,0,0.239693,0.239693,0.993659
1,Selected,1,0.123682,0.123682,0.741241


In [79]:
model_df = pd.DataFrame([model_data])
baseline_df = pd.DataFrame([baseline_data])

display_df = pd.DataFrame(index=model_df.index)
display_df['category'] = model_df['category']

for col in model_df.columns:
    if col == 'category':
        continue
    model_values = model_df[col]
    baseline_values = baseline_df[col]
    delta = model_values - baseline_values
    if str(col) in ['2025', 'final']:
        display_df[col] = model_values.mul(100).round(1).apply(lambda x: f"{x:.0f}%") + \
                        delta.mul(100).round(1).apply(lambda x: f"({x:+.0f}%)")

    else:
        display_df[col] = model_values.round(2).astype(str) + \
                            delta.round(2).apply(lambda x: f" ({x:+.2f})")
                            
display_df = display_df.drop(columns=['baseline'])
display_df

,category,2025,final,sharpe
0,Selected,24%(+12%),24%(+12%),0.99 (+0.25)
